# Лабораторная работа №14. От URL до страницы — итоговая сеть

Перед вами сломанный стенд. Не чините всё подряд. Идите по цепочке: name resolution → routing → ICMP/traceroute → TCP → HTTP. В стенде спрятано **5 неисправностей**.

## Схема

![Capstone](assets/capstone.svg)

Цель: с `pc1` получить страницу `http://app.lab.example:8080/`.

Адреса:

| Узел | Интерфейс | Адрес |
|---|---|---|
| pc1 | eth1 | `10.14.1.10/24` |
| r1 | eth1 | `10.14.1.1/24` |
| r1 | eth2 | `10.14.12.1/30` |
| r2 | eth1 | `10.14.12.2/30` |
| r2 | eth2 | `10.14.20.1/24` |
| dns1 | eth1 | `10.14.20.53/24` |
| r2 | eth3 | `10.14.30.1/24` |
| web1 | eth1 | `10.14.30.80/24` |

## Правила capstone

1. Сначала наблюдение, потом изменение.
2. После каждого fix повторяйте только тот test, который раньше падал.
3. Записывайте причину каждой неисправности, а не только команду исправления.
4. CHECK запускайте в конце или когда хотите проверить прогресс.

### Q1. Какой самый первый тест вы выберете для URL `http://app.lab.example:8080/`: DNS query, ping, traceroute, TCP connect или curl? Обоснуйте порядок.

`введите ваш ответ`

## 1. Начальная диагностика на pc1

```bash
ip -br addr
ip -4 route
cat /etc/resolv.conf
getent hosts app.lab.example
curl -v --connect-timeout 2 http://app.lab.example:8080/
```

Не исправляйте всё сразу. Если имя не разрешается, отдельно проверьте сам DNS server по IP:

```bash
ip route get 10.14.20.53
dig @10.14.20.53 app.lab.example A
```

### Q2. Почему `dig @10.14.20.53 ...` и обычный `getent hosts ...` проверяют разные части DNS-path?

`введите ваш ответ`

## 2. Идите hop-by-hop

Полезные команды:

```bash
# pc1
ip route get 10.14.20.53
ip route get 10.14.30.80
traceroute -I 10.14.20.53
traceroute -I 10.14.30.80

# r1 / r2
show ip route
show ip route 10.14.20.53
show ip route 10.14.30.80
show ip route 10.14.1.10
```

В этой работе **нет** одной команды, которая диагностирует всё.

### Q3. Если DNS Request дошёл до dns1, но Reply не возвращается на pc1, какой класс проблемы нужно искать: DNS record или обратную маршрутизацию?

`введите ваш ответ`

## 3. Подсказка по ожидаемой конечной модели

После исправления должны существовать:

- на `pc1` более специфичные routes к `10.14.20.0/24` и `10.14.30.0/24` через R1;
- на `r1` routes к обеим удалённым LAN через R2;
- на `r2` обратный route к `10.14.1.0/24` через R1;
- системный resolver `pc1` должен использовать `10.14.20.53`;
- `web1` должен реально слушать TCP/8080.

Это не список готовых команд: найдите каждое нарушение через наблюдения.

## 4. DNS должен привести к следующему шагу

Когда routing к DNS исправлен:

```bash
dig @10.14.20.53 app.lab.example A
```

Ожидаемый application result DNS: `10.14.30.80`. После этого проверяйте именно reachability этого адреса и TCP/8080.

## 5. TCP/HTTP

На `pc1`:

```bash
ping -c 2 10.14.30.80
nc -vz 10.14.30.80 8080
curl -v --connect-timeout 2 http://app.lab.example:8080/
```

Если IP достижим, а TCP connect нет — не возвращайтесь к ARP/DNS без причины. Ищите service/listener/firewall layer.

In [ ]:
%%capture_traffic pc1:eth1 --filter "dns or icmp or tcp.port == 8080" --timeout 20 --packets 400 --save captures/capstone.pcap
import time
print("После исправлений выполните dig, ping и curl с pc1")
time.sleep(18)

In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display
viewer = PcapViewer("captures/capstone.pcap", "dns or icmp or tcp.port == 8080", limit=120)
frames = viewer.packets()
display(frames)

### Q4. Перечислите найденные неисправности в порядке диагностики: симптом → наблюдение → причина → исправление. Не перечисляйте только команды.

`введите ваш ответ`

## Финальный CHECK

Успех означает не только `curl 200`: checker отдельно проверяет routing state, DNS answer и HTTP body. Это не позволяет случайно «обойти» одну неисправность костылём на другом уровне.

## Итог

Troubleshooting сети — это управление гипотезами. URL превращается в DNS query, адрес — в routing/L2 delivery, затем TCP создаёт transport state и только после этого HTTP получает шанс ответить.